# 🧩 Mini-projet : Agent IA avec MCP + Gemini (Google Colab)

**But :** construire un **agent** qui orchestre **plusieurs serveurs MCP** en laissant
le **LLM (Gemini)** décider quels outils appeler (aucun flux codé en dur).

**Thème choisi : « Assistant d'espace de travail »**
On compose **3 serveurs MCP** :
1. 🗂️ **filesystem** *(serveur tiers, via npx)* — lire/écrire des fichiers ;
2. 🌿 **git** *(serveur tiers, via python)* — statut/historique d'un dépôt ;
3. 🛠️ **workspace_ops** *(serveur MAISON en FastMCP)* — outils personnalisés (résumé de lignes…).

> ✅ L'énoncé demande **au moins 2 serveurs tiers** : ici filesystem + git (+ 1 maison).
>
> ⚠️ **Prérequis :** une **clé API Google (Gemini)** gratuite depuis Google AI Studio
> (https://aistudio.google.com/app/apikey). On la renseigne à l'étape 1.
>
> 👉 Exécutez les cellules **de haut en bas**.

*Pour changer de thème, il suffit d'adapter les outils du serveur maison (étape 4)
et les serveurs branchés (étape 5).*

## 0) Installation des dépendances

- `langchain` / `langgraph` : construction de l'agent ;
- `langchain-google-genai` + `google-genai` : connexion à **Gemini** ;
- `langchain-mcp-adapters` : expose les outils MCP comme outils LangChain ;
- `fastmcp` : pour écrire notre **serveur MCP maison** ;
- `mcp-server-git` : le **serveur git** (absent du brief d'origine, on l'ajoute !) ;
- `nest_asyncio` : indispensable pour utiliser `await` dans Colab.

In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "fastmcp>=2.0.0" \
  "mcp-server-git" \
  "nest_asyncio"

## 1) Clé API Gemini

On stocke la clé dans la variable d'environnement `GOOGLE_API_KEY`
(c'est le nom que `langchain-google-genai` lit automatiquement).

Deux méthodes : via les **Secrets de Colab** (🔑 icône clé à gauche, ajouter
`GOOGLE_API_KEY`), sinon saisie manuelle sécurisée.

In [ ]:
import os

try:
    # Méthode recommandée dans Colab : Secrets (menu 🔑 à gauche).
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("Clé chargée depuis les Secrets Colab ✅")
except Exception:
    # Repli : saisie masquée (la clé n'apparaît pas à l'écran).
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Entrez votre clé API Google (Gemini) : ")
    print("Clé chargée manuellement ✅")

## 2) Vérifier Node / npx

Le serveur **filesystem** est un paquet **Node** lancé via `npx`. Colab l'inclut
généralement déjà. On vérifie ; en cas d'absence, on installe.

In [ ]:
!node --version || (apt-get -qq update && apt-get -qq install -y nodejs npm)
!npx --version

## 3) Préparer l'espace de travail + activer `await`

On crée un dossier `WORKDIR` que les serveurs filesystem et git manipuleront.
On en fait un **vrai dépôt git** (avec un commit) pour que les outils git aient
quelque chose à montrer.

`nest_asyncio.apply()` permet d'utiliser `await` directement dans les cellules Colab
(sinon la boucle d'événements déjà active provoque des erreurs).

In [ ]:
import nest_asyncio, subprocess, os
nest_asyncio.apply()   # autorise await dans Colab

WORKDIR = "/content/workspace"
os.makedirs(WORKDIR, exist_ok=True)

# Deux fichiers d'exemple pour donner du "grain à moudre" à l'agent.
with open(os.path.join(WORKDIR, "README.md"), "w", encoding="utf-8") as f:
    f.write("# Projet Demo\n\nCeci est un espace de travail de démonstration pour l'agent MCP.\n")
with open(os.path.join(WORKDIR, "notes.txt"), "w", encoding="utf-8") as f:
    f.write("ligne 1\n\nligne 3 (la 2 est vide)\nligne 4\n")

# Initialisation d'un dépôt git avec un premier commit (pour les outils git).
def sh(cmd):
    return subprocess.run(cmd, cwd=WORKDIR, shell=True, capture_output=True, text=True).stdout

sh("git init -q")
sh('git config user.email "demo@example.com"')
sh('git config user.name "Demo"')
sh("git add -A")
sh('git commit -q -m "commit initial"')

print("Espace de travail prêt :", WORKDIR)
print(sh("git log --oneline"))

## 4) Écrire le serveur MCP maison (FastMCP)

On écrit un fichier Python `custom_mcp_server.py` qui expose des outils via `@mcp.tool`,
puis se lance en `stdio` (le transport que l'agent utilise pour parler au serveur).

> ⚠️ **Bug corrigé du brief :** le code d'origine mettait des docstrings `"""..."""`
> **à l'intérieur** d'une chaîne elle-même délimitée par `"""..."""` → Python fermait
> la chaîne trop tôt (erreur de syntaxe). Ici, on délimite le code du serveur avec des
> **triple-guillemets simples** `'''...'''`, ce qui laisse les docstrings `"""` tranquilles.

In [ ]:
from pathlib import Path

# On délimite le code du serveur avec '''  pour ne pas casser les docstrings """...""".
server_code = r'''
from fastmcp import FastMCP
from typing import Dict, List

# Nom du serveur (apparaîtra en préfixe des outils : workspace_ops__...).
mcp = FastMCP(name="workspace_ops")

@mcp.tool
def ping() -> str:
    """Vérifie que le serveur répond (renvoie 'pong')."""
    return "pong"

@mcp.tool
def summarize_lines(lines: List[str]) -> Dict[str, int]:
    """Compte le nombre total de lignes et le nombre de lignes non vides."""
    total = len(lines)
    nonempty = sum(1 for l in lines if l.strip())
    return {"total_lines": total, "nonempty_lines": nonempty}

@mcp.tool
def word_count(text: str) -> int:
    """Compte le nombre de mots dans un texte."""
    return len(text.split())

if __name__ == "__main__":
    # Transport stdio : l'agent lance ce fichier comme sous-processus et dialogue via stdin/stdout.
    mcp.run(transport="stdio")
'''

server_path = Path("/content/custom_mcp_server.py")
server_path.write_text(server_code, encoding="utf-8")
print("Serveur maison écrit :", server_path)

## 5) Connecter les serveurs MCP

`MultiServerMCPClient` enregistre nos 3 serveurs. `tool_name_prefix=True` préfixe
chaque outil par le nom de son serveur (ex. `filesystem__read_file`), pour éviter
les collisions de noms.

> ⚠️ **Point clé (corrigé) :** `client.get_tools()` est une **coroutine** → on doit
> l'appeler avec `await` (le `run_until_complete(...)` du brief plante dans Colab).

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_connections = {
    # 1) Serveur TIERS : système de fichiers (Node via npx). Il n'agit QUE dans WORKDIR.
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    # 2) Serveur TIERS : git (module Python). Pointe sur notre dépôt de démo.
    "git": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "mcp_server_git", "--repository", WORKDIR],
    },
    # 3) Serveur MAISON : nos outils personnalisés écrits à l'étape 4.
    "workspace_ops": {
        "transport": "stdio",
        "command": "python",
        "args": [str(server_path)],
    },
}

client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)

# get_tools() est ASYNCHRONE -> on utilise await (possible grâce à nest_asyncio).
tools = await client.get_tools()

print("Nombre total d'outils découverts :", len(tools))
for t in tools:
    print(" -", t.name)

## 6) Construire l'agent Gemini

On crée le modèle **Gemini** puis un **agent ReAct** (`create_react_agent`) : à chaque
tour, le LLM lit la question + la liste des outils et **décide lui-même** lequel appeler.

> 💡 `gemini-2.5-flash` est rapide et économique. Vous pouvez changer le nom du modèle
> (voir https://ai.google.dev/gemini-api/docs/models).

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

# Le "cerveau" : lit GOOGLE_API_KEY dans l'environnement (étape 1).
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# Consigne système : le LLM choisit les outils, cite les fichiers/sources, reste concis.
SYSTEME = (
    "Tu es un assistant d'espace de travail. Tu peux utiliser des outils MCP "
    "(système de fichiers, git, et opérations personnalisées). "
    "Choisis toi-même les outils nécessaires pour répondre. "
    "Cite les fichiers ou sources utilisés. Réponds de façon concise (2 à 4 phrases). "
    "Si l'information manque, dis-le et propose une question de suivi."
)

# L'agent = modèle + outils MCP + consigne.
agent = create_react_agent(model, tools, prompt=SYSTEME)
print("✅ Agent prêt avec", len(tools), "outils.")

## 7) Lancer l'agent sur des requêtes de démonstration

Chaque requête peut nécessiter **plusieurs serveurs**. La fonction `run_agent`
affiche, pour chaque question : **les appels d'outils** effectués (nom + arguments),
les **résultats d'outils**, puis la **réponse finale**.

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage

async def run_agent(question: str):
    print("=" * 72)
    print("❓", question)
    # ainvoke : version asynchrone (les outils MCP sont asynchrones).
    result = await agent.ainvoke({"messages": [{"role": "user", "content": question}]})

    # On parcourt l'historique des messages pour montrer ce que l'agent a fait.
    for m in result["messages"]:
        # a) Le LLM a demandé d'appeler un ou plusieurs outils ?
        if isinstance(m, AIMessage) and m.tool_calls:
            for tc in m.tool_calls:
                print("   🔧 appel:", tc["name"], "| args:", tc["args"])
        # b) Un outil a renvoyé un résultat ?
        elif isinstance(m, ToolMessage):
            apercu = str(m.content).replace("\n", " ")[:120]
            print("   📥 résultat", f"({m.name}):", apercu)

    # Le dernier message est la réponse finale de l'agent.
    print("💬 Réponse :", result["messages"][-1].content)


# Trois questions qui sollicitent des serveurs différents :
requetes = [
    "Liste les fichiers de l'espace de travail, puis lis README.md et résume-le.",  # filesystem
    "Quel est l'historique git (les commits) du dépôt ?",                            # git
    "Lis notes.txt puis, avec l'outil summarize_lines, dis combien de lignes non vides il contient.",  # filesystem + maison
]

async def demo():
    for q in requetes:
        try:
            await run_agent(q)
        except Exception as e:
            print("⚠️ Erreur :", type(e).__name__, "-", e)
    print("=" * 72)

await demo()

## 8) Aller plus loin

- **Changer de thème** : remplacez les outils du serveur maison (étape 4) — ex.
  `get_menu()` / `price_order(items)` pour un « Assistant café », ou
  `format_markdown(text)` / `extract_entities(text)` pour un « Assistant de recherche ».
- **Ajouter un serveur MCP tiers** : cherchez dans la liste officielle des serveurs MCP
  et ajoutez une entrée dans `mcp_connections`.
- **Filtrer les outils** : passez `server_name="git"` à `get_tools()` pour ne récupérer
  que les outils d'un serveur donné.
- **Fiabilité** : si Gemini n'appelle pas le bon outil, précisez la consigne système
  ou nommez explicitement l'outil dans la question.